#### Introduction

In [1]:
import pandas as pd
import os

##### Importing the data

In [2]:
excel_files = [file for file in os.listdir('./NSE Data Collection') if file.endswith('.xlsx')]
csv_files = [file for file in os.listdir('./NSE Data Collection') if file.endswith('.csv')]

consolidated_data = pd.DataFrame()

for file in excel_files:
    df=pd.read_excel(os.path.join('./NSE Data Collection', file))
    df.rename(columns=str.upper, inplace=True)
    df.rename(columns={'ADJUST':'ADJUSTED PRICE'}, inplace=True)
    consolidated_data = pd.concat([consolidated_data, df], ignore_index=True)

for file in csv_files:
    df=pd.read_csv(os.path.join('./NSE Data Collection', file))
    df.rename(columns=str.upper, inplace=True)  
    df.rename(columns={'ADJUST':'ADJUSTED PRICE'}, inplace=True)
    consolidated_data = pd.concat([consolidated_data, df], ignore_index=True)

ref_file = "Reference File.xlsx"

consolidated_data.head()

,DATE,CODE,NAME,12M LOW,12M HIGH,DAY LOW,DAY HIGH,DAY PRICE,PREVIOUS,CHANGE,CHANGE%,VOLUME,ADJUSTED PRICE
0,2013-01-02 00:00:00,EGAD,Eaagads Ltd,8.2,14.0,25.0,25.0,25.0,25,-,-,-,-
1,2013-01-02 00:00:00,KUKZ,Kakuzi Plc,300.0,423.5,67.5,67.5,67.5,72,-4.5,0.0625,300,-
2,2013-01-02 00:00:00,KAPC,Kapchorua Tea Kenya Plc,59.0,95.0,118.0,118.0,118.0,118,-,-,-,59
3,2013-01-02 00:00:00,LIMT,Limuru Tea Plc,385.0,475.0,430.0,430.0,430.0,430,-,-,-,215
4,2013-01-02 00:00:00,SASN,Sasini Plc,14.8,19.6,11.7,12.05,11.9,11.7,0.2,0.0171,14500,-


#### Data Cleaning

##### >>> Cleaning the reference file

In [3]:
ref_df = pd.read_excel(ref_file)
ref_df.head()

,SECTOR,CODE,NAME
0,Agricultural,EGAD,Eaagads Ltd
1,Agricultural,KUKZ,Kakuzi Plc
2,Agricultural,KAPC,Kapchorua Tea Kenya Plc
3,Agricultural,LIMT,Limuru Tea Plc
4,Agricultural,SASN,Sasini Plc


In [4]:
ref_df.shape

(500, 3)

In [5]:
ref_df.drop_duplicates(inplace=True)

In [6]:
ref_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 99 entries, 0 to 418
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   SECTOR  99 non-null     object
 1   CODE    99 non-null     object
 2   NAME    98 non-null     object
dtypes: object(3)
memory usage: 3.1+ KB


In [7]:
ref_df['SECTOR'].value_counts()

SECTOR
Commercial and Services             13
Construction and Allied             13
Banking                             12
Manufacturing and Allied            10
Indices                             10
Energy and Petroleum                 9
Insurance                            7
Agricultural                         6
Investment                           5
Real Estate Investment Trusts        4
Exchange Traded Funds                2
^NASI                                1
Investment Services                  1
Telecommunication and Technology     1
^N20I                                1
^FNK25                               1
^FNK15                               1
Automobiles and Accessories          1
Telecommunication                    1
Name: count, dtype: int64

##### >>> Cleaning the NSE file

In [8]:
consolidated_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 246662 entries, 0 to 246661
Data columns (total 13 columns):
 #   Column          Non-Null Count   Dtype 
---  ------          --------------   ----- 
 0   DATE            246660 non-null  object
 1   CODE            246660 non-null  object
 2   NAME            246660 non-null  object
 3   12M LOW         246660 non-null  object
 4   12M HIGH        246660 non-null  object
 5   DAY LOW         246660 non-null  object
 6   DAY HIGH        246660 non-null  object
 7   DAY PRICE       246660 non-null  object
 8   PREVIOUS        246660 non-null  object
 9   CHANGE          246660 non-null  object
 10  CHANGE%         246660 non-null  object
 11  VOLUME          246660 non-null  object
 12  ADJUSTED PRICE  246660 non-null  object
dtypes: object(13)
memory usage: 24.5+ MB


In [9]:
consolidated_data.drop_duplicates(inplace=True)

In [10]:
consolidated_data.isnull().sum()

DATE              1
CODE              1
NAME              1
12M LOW           1
12M HIGH          1
DAY LOW           1
DAY HIGH          1
DAY PRICE         1
PREVIOUS          1
CHANGE            1
CHANGE%           1
VOLUME            1
ADJUSTED PRICE    1
dtype: int64

In [11]:
consolidated_data = consolidated_data[consolidated_data.notna()]

In [12]:
consolidated_data.head()

,DATE,CODE,NAME,12M LOW,12M HIGH,DAY LOW,DAY HIGH,DAY PRICE,PREVIOUS,CHANGE,CHANGE%,VOLUME,ADJUSTED PRICE
0,2013-01-02 00:00:00,EGAD,Eaagads Ltd,8.2,14.0,25.0,25.0,25.0,25,-,-,-,-
1,2013-01-02 00:00:00,KUKZ,Kakuzi Plc,300.0,423.5,67.5,67.5,67.5,72,-4.5,0.0625,300,-
2,2013-01-02 00:00:00,KAPC,Kapchorua Tea Kenya Plc,59.0,95.0,118.0,118.0,118.0,118,-,-,-,59
3,2013-01-02 00:00:00,LIMT,Limuru Tea Plc,385.0,475.0,430.0,430.0,430.0,430,-,-,-,215
4,2013-01-02 00:00:00,SASN,Sasini Plc,14.8,19.6,11.7,12.05,11.9,11.7,0.2,0.0171,14500,-


In [13]:
ref_df.head()

,SECTOR,CODE,NAME
0,Agricultural,EGAD,Eaagads Ltd
1,Agricultural,KUKZ,Kakuzi Plc
2,Agricultural,KAPC,Kapchorua Tea Kenya Plc
3,Agricultural,LIMT,Limuru Tea Plc
4,Agricultural,SASN,Sasini Plc


In [14]:
ref_df['CODE'] = ref_df['CODE'].astype(str).str.strip().str.upper()
consolidated_data['CODE'] = consolidated_data['CODE'].astype(str).str.strip().str.upper()

sector_map = (
    ref_df.drop_duplicates(subset='CODE')
    .set_index('CODE')['SECTOR']
)

consolidated_data['SECTOR'] = consolidated_data['CODE'].map(sector_map)
consolidated_data.head()

,DATE,CODE,NAME,12M LOW,12M HIGH,DAY LOW,DAY HIGH,DAY PRICE,PREVIOUS,CHANGE,CHANGE%,VOLUME,ADJUSTED PRICE,SECTOR
0,2013-01-02 00:00:00,EGAD,Eaagads Ltd,8.2,14.0,25.0,25.0,25.0,25,-,-,-,-,Agricultural
1,2013-01-02 00:00:00,KUKZ,Kakuzi Plc,300.0,423.5,67.5,67.5,67.5,72,-4.5,0.0625,300,-,Agricultural
2,2013-01-02 00:00:00,KAPC,Kapchorua Tea Kenya Plc,59.0,95.0,118.0,118.0,118.0,118,-,-,-,59,Agricultural
3,2013-01-02 00:00:00,LIMT,Limuru Tea Plc,385.0,475.0,430.0,430.0,430.0,430,-,-,-,215,Agricultural
4,2013-01-02 00:00:00,SASN,Sasini Plc,14.8,19.6,11.7,12.05,11.9,11.7,0.2,0.0171,14500,-,Agricultural


In [15]:
consolidated_data.to_excel('consolidated_data.xlsx', index=False)

In [16]:
df = consolidated_data[['DATE', 'CODE', 'SECTOR', 'DAY PRICE', 'PREVIOUS']]
df.head()

,DATE,CODE,SECTOR,DAY PRICE,PREVIOUS
0,2013-01-02 00:00:00,EGAD,Agricultural,25.0,25
1,2013-01-02 00:00:00,KUKZ,Agricultural,67.5,72
2,2013-01-02 00:00:00,KAPC,Agricultural,118.0,118
3,2013-01-02 00:00:00,LIMT,Agricultural,430.0,430
4,2013-01-02 00:00:00,SASN,Agricultural,11.9,11.7


In [17]:
consolidated_data.shape

(246592, 14)

In [18]:
df['MONTH'] = pd.to_datetime(df['DATE']).dt.month
df['YEAR'] = pd.to_datetime(df['DATE']).dt.year

C:\Users\Naftal\AppData\Local\Temp\ipykernel_9168\90839473.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['MONTH'] = pd.to_datetime(df['DATE']).dt.month
C:\Users\Naftal\AppData\Local\Temp\ipykernel_9168\90839473.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['YEAR'] = pd.to_datetime(df['DATE']).dt.year


In [19]:
df.head()

,DATE,CODE,SECTOR,DAY PRICE,PREVIOUS,MONTH,YEAR
0,2013-01-02 00:00:00,EGAD,Agricultural,25.0,25,1.0,2013.0
1,2013-01-02 00:00:00,KUKZ,Agricultural,67.5,72,1.0,2013.0
2,2013-01-02 00:00:00,KAPC,Agricultural,118.0,118,1.0,2013.0
3,2013-01-02 00:00:00,LIMT,Agricultural,430.0,430,1.0,2013.0
4,2013-01-02 00:00:00,SASN,Agricultural,11.9,11.7,1.0,2013.0


#### Encoding the DataFrame

In [20]:

df = pd.get_dummies(df, columns=["CODE"], prefix="CODE")
df = pd.get_dummies(df, columns=["SECTOR"], prefix="SECTOR")
df.head()


,DATE,DAY PRICE,PREVIOUS,MONTH,YEAR,CODE_ABSA,CODE_ALP,CODE_AMAC,CODE_ARM,CODE_BAMB,...,SECTOR_Insurance,SECTOR_Investment,SECTOR_Investment Services,SECTOR_Manufacturing and Allied,SECTOR_Real Estate Investment Trusts,SECTOR_Telecommunication and Technology,SECTOR_^FNK15,SECTOR_^FNK25,SECTOR_^N20I,SECTOR_^NASI
0,2013-01-02 00:00:00,25.0,25,1.0,2013.0,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,2013-01-02 00:00:00,67.5,72,1.0,2013.0,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,2013-01-02 00:00:00,118.0,118,1.0,2013.0,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,2013-01-02 00:00:00,430.0,430,1.0,2013.0,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,2013-01-02 00:00:00,11.9,11.7,1.0,2013.0,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [21]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

for column in ["DAY PRICE", "PREVIOUS"]:
    df[column] = pd.to_numeric(
        df[column].astype(str).str.replace(",", "", regex=False),
        errors="coerce"
    )

df.dropna(subset=["DAY PRICE", "PREVIOUS"], inplace=True)
df["PREVIOUS"] = df["PREVIOUS"].astype(float)
# df[["DAY PRICE", "PREVIOUS"]] = scaler.fit_transform(df[["DAY PRICE", "PREVIOUS"]])
df[["DAY PRICE", "PREVIOUS"]]

,DAY PRICE,PREVIOUS
0,25.00,25.00
1,67.50,72.00
2,118.00,118.00
3,430.00,430.00
4,11.90,11.70
...,...,...
246657,224.15,222.82
246658,255.11,253.38
246659,1129.57,1129.74
246660,5090.52,5037.88


In [22]:
df.head()

,DATE,DAY PRICE,PREVIOUS,MONTH,YEAR,CODE_ABSA,CODE_ALP,CODE_AMAC,CODE_ARM,CODE_BAMB,...,SECTOR_Insurance,SECTOR_Investment,SECTOR_Investment Services,SECTOR_Manufacturing and Allied,SECTOR_Real Estate Investment Trusts,SECTOR_Telecommunication and Technology,SECTOR_^FNK15,SECTOR_^FNK25,SECTOR_^N20I,SECTOR_^NASI
0,2013-01-02 00:00:00,25.0,25.0,1.0,2013.0,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,2013-01-02 00:00:00,67.5,72.0,1.0,2013.0,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,2013-01-02 00:00:00,118.0,118.0,1.0,2013.0,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,2013-01-02 00:00:00,430.0,430.0,1.0,2013.0,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,2013-01-02 00:00:00,11.9,11.7,1.0,2013.0,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [23]:
X = df.drop(columns=['DAY PRICE','DATE'])
y = df['DAY PRICE']
X, y

(        PREVIOUS  MONTH    YEAR  CODE_ABSA  CODE_ALP  CODE_AMAC  CODE_ARM  \
 0          25.00    1.0  2013.0      False     False      False     False   
 1          72.00    1.0  2013.0      False     False      False     False   
 2         118.00    1.0  2013.0      False     False      False     False   
 3         430.00    1.0  2013.0      False     False      False     False   
 4          11.70    1.0  2013.0      False     False      False     False   
 ...          ...    ...     ...        ...       ...        ...       ...   
 246657    222.82    6.0  2026.0      False     False      False     False   
 246658    253.38    6.0  2026.0      False     False      False     False   
 246659   1129.74    6.0  2026.0      False     False      False     False   
 246660   5037.88    6.0  2026.0      False     False      False     False   
 246661   3349.51    6.0  2026.0      False     False      False     False   
 
         CODE_BAMB  CODE_BAT  CODE_BKG  ...  SECTOR_Insurance 

In [24]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1)

print('Training set size:', X_train.shape)
print('Test set size:', X_test.shape)


Training set size: (172605, 108)
Test set size: (73974, 108)


#### Linear Regression Model

In [25]:
from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(X_train, y_train)
predictions = model.predict(X_test)

In [ ]:
intercept = model.intercept_
coefficients = model.coef_

print('Intercept:', intercept)
print('Coefficients:')
print(coefficients)

Intercept: 321.0336310221462
Coefficients:
[ 9.66524378e-01 -1.02443126e-01 -1.53025474e-01 -2.39096996e+00
 -2.53963469e+00 -9.26138905e-01 -2.32509372e+00  5.08047730e-01
  1.43869970e+01 -1.27351597e+00 -1.23049492e+00 -3.21238696e+00
 -3.03204323e+00 -3.39751075e+00 -5.18558657e+00 -3.44019733e+00
 -2.31295606e+00 -1.26519142e+00 -9.76944789e+00 -1.05835048e+00
  1.53667990e+00  3.08141856e+00 -6.73075446e+00 -1.40183047e+00
 -1.36731282e+00 -3.40435549e+00 -9.11920973e+00 -4.18575114e+00
  3.96113775e+01 -1.05216425e+01 -5.40650067e-01 -2.38115146e+00
 -1.05475597e+01 -4.00478147e-01  7.62735338e+00 -2.82688008e+00
 -1.41535002e+00 -1.40054594e+00 -1.73472207e+00 -3.23164952e+00
 -4.02152278e-01 -1.36324792e+00 -1.46844673e+00 -1.44468205e+00
 -1.21712836e+00  2.85722095e+00  4.04303721e+01 -1.99847650e+00
 -3.19127119e+00  1.01502818e+01 -1.23069933e+00 -4.47628787e+00
  0.00000000e+00 -2.57592073e+00 -9.91813406e-01 -1.38947164e+00
  2.25102353e+00 -5.39902640e+00 -1.05534518e+0

In [27]:
# R-squared score on the test data
r_squared = model.score(X_test, y_test)
print("R-squared:", r_squared)

R-squared: 0.9903587410322517


In [28]:
results_df = pd.DataFrame({
	'Actual': y_test,
	'Predicted': pd.Series(predictions, index=y_test.index)
})

results_df["Percentage Error"] = ((results_df['Predicted'] - results_df['Actual']) / results_df['Actual']) * 100
print(results_df.head())


        Actual   Predicted  Percentage Error
70473     1.30    1.718781         32.213917
170166    1.15    1.322646         15.012717
146374   15.55   18.350049         18.006748
34878     3.55    4.527044         27.522370
28286   178.79  177.226902         -0.874265


In [33]:
results_df.head(20)

,Actual,Predicted,Percentage Error
70473,1.30,1.718781,32.213917
170166,1.15,1.322646,15.012717
146374,15.55,18.350049,18.006748
34878,3.55,4.527044,27.522370
28286,178.79,177.226902,-0.874265
30854,320.00,318.127121,-0.585275
202454,364.00,361.811956,-0.601111
188569,77.00,77.257637,0.334594
107553,28.75,29.420591,2.332492
139049,46.80,47.733377,1.994395
